# Large Language Models and How to Instruction Tune Them (in a Sustainable Way)

This is an implementation for training and using a Large Language Model (based on [LLaMA](https://ai.meta.com/blog/large-language-model-llama-meta-ai/)) with instructions in order to solve the linguistic tasks. In this lab, we will see how to encode datasets from any format to a sequence to sequence format, train the model using [Q-LoRA](https://arxiv.org/abs/2305.14314), perform the inference using the previous trained model for generating answers to instructions, and finally, how to encode back the data to the original format.. all of it using the only available *T4 GPU with 15GB from Google Colab*.

In [ ]:
# You need to install many libraries!
# PEFT for model parameters efficient finetuning
# SententencePiece for Byte Level Tokenizer
# Accellerate to optimize model over GPU
# bitsandbytes for using quantization!
# datasets for HuggingFace datasets download and processing support
# Moreover you should install the Transformers library (already installed for you on Colab!)

#!pip install -U bitsandbytes
#!pip install peft
#!pip install sentencepiece
#!pip install accelerate
#!pip install datasets
#!pip install -U transformers

# Finetuning data: compare to pretraining and basic preparation

In [ ]:
#!pip install jsonlines
#!pip install datasets

## Look at pretraining data set

In [ ]:
import jsonlines
import itertools
import pandas as pd
from pprint import pprint

import datasets
from datasets import load_dataset
pretraining_dataset = load_dataset(
    "allenai/c4",
    "en",
    split="train",
    streaming=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

In [ ]:
n = 5
print("Pretrained dataset:")
top_n = itertools.islice(pretraining_dataset, n)
for i in top_n:
  print(i)

Pretrained dataset:
{'text': 'Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.', 'timestamp': '2019-04-25 12:57:54', 'url': 'https://klyq.com/beginners-bbq-class-taking-place-in-missoula/'}
{'text': 'Discussion in \'Mac OS X Lion (10.7)\' started by axboi87, Jan 20, 2012.\nI\'ve got a 500gb intern

You do not need any template for continue the training of the model over plain text! Just follow the standard Auto-Regressive Conditional approach! P(y| y<t)

The only things to add are the **Begin of Sentence \<BOS\>** and the **End of Sentence \<EOS\>** tokens! Be carefull they are Model Specific!

In [ ]:
#Create dataset for continue pre-training.
pretrain_text = []
template_pretrain = "<|begin_of_text|>{text}<|end_of_text|>"

top_n = itertools.islice(pretraining_dataset, 100)
for row in top_n:
  pretrain_text.append(template_pretrain.format(text=row["text"]))

print(pretrain_text[0])

<|begin_of_text|>Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.
The cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.<|end_of_text|>


## Finetuning dataset format

In [ ]:
instruction_dataset_df = load_dataset("kotzeje/lamini_docs.jsonl", split="train", streaming=True)

n = 5
print("Instruction dataset:")
top_n = itertools.islice(instruction_dataset_df, n)
for i in top_n:
  print(i)

README.md:   0%|          | 0.00/481 [00:00<?, ?B/s]

Instruction dataset:
{'question': 'How can I evaluate the performance and quality of the generated text from Lamini models?', 'answer': "There are several metrics that can be used to evaluate the performance and quality of generated text from Lamini models, including perplexity, BLEU score, and human evaluation. Perplexity measures how well the model predicts the next word in a sequence, while BLEU score measures the similarity between the generated text and a reference text. Human evaluation involves having human judges rate the quality of the generated text based on factors such as coherence, fluency, and relevance. It is recommended to use a combination of these metrics for a comprehensive evaluation of the model's performance."}
{'question': "Can I find information about the code's approach to handling long-running tasks and background jobs?", 'answer': 'Yes, the code includes methods for submitting jobs, checking job status, and retrieving job results. It also includes a method fo

## Various ways of formatting your data

In order to use Instructions as a template for a Supervised Fine Tuning Approach, you should use the **appropriate LLM prompt template**, including the answer the model should provide!
[https://github.com/oobabooga/text-generation-webui/tree/main/instruction-templates](https://github.com/oobabooga/text-generation-webui/tree/main/instruction-templates)

In [ ]:
prompt_template_LLaMa = '''<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a very helpful assistant. Answer questions as best you can.<|eot_id|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{answer}<|eot_id|><|end_of_text|>'''

In [ ]:
text_with_prompt_template=[]
for row in instruction_dataset_df:
  question = row["question"]
  answer = row["answer"]

  text_with_prompt_template.append(prompt_template_LLaMa.format(question=question, answer=answer))
print(text_with_prompt_template[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a very helpful assistant. Answer questions as best you can.<|eot_id|><|start_header_id|>user<|end_header_id|>

How can I evaluate the performance and quality of the generated text from Lamini models?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There are several metrics that can be used to evaluate the performance and quality of generated text from Lamini models, including perplexity, BLEU score, and human evaluation. Perplexity measures how well the model predicts the next word in a sequence, while BLEU score measures the similarity between the generated text and a reference text. Human evaluation involves having human judges rate the quality of the generated text based on factors such as coherence, fluency, and relevance. It is recommended to use a combination of these metrics for a comprehensive evaluation of the model's performance.<|eot_id|><|end_of_text|>


The tutorial is split into 4 steps, reflecting the aforementioned process:
- Step 1 - Encoding the data
- **Step 2 - Training the LLaMA model**
- Step 3 - Inference: generating answers
- Step 4 - Evaluate quality of generation

The input of the Notebook is the data we generated previously, in order to train the model.

The "output" will be the trained LLaMA model: we will save it on Google Colab and (optional) upload it to the HuggingFace Hub.

Please make sure you are using the runtime environment according to the following settings: **T4 GPU RUNTIME**


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import transformers
from transformers import LlamaTokenizer, LlamaForCausalLM
import csv
import json
import torch
from datasets import load_dataset
import pandas as pd

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

## The Main part
We first load the data that was already prepared to contain maximum 200 examples.

In [ ]:
DEVICE = "cuda"
TOKENIZER_MODEL = "1024m/Llama-3.2-3B-Instruct-Base"
BASE_MODEL = "1024m/Llama-3.2-3B-Instruct-Base"

OUTPUT_DIR = f"HCNLP-Llama-3.2-3B-Instruct-LLamini"

CUTOFF_LEN = 512
CUT_INPUT_CHAR_LENGTH = 1200

In [ ]:
# PREPARE DATA
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

from datasets import Dataset
def tokenize(prompt):
    print(prompt["text"])
    result = tokenizer(
        prompt["text"],
        truncation=True,
        max_length=512,
        padding=False,
        return_tensors=None,
    )
    if (
        result["input_ids"][-1] != tokenizer.eos_token_id
        and len(result["input_ids"]) < 512
    ):
        result["input_ids"].append(tokenizer.eos_token_id)
        result["attention_mask"].append(1)

    result["labels"] = result["input_ids"].copy()
    return result


def gen():
  for t in text_with_prompt_template[:200]:
    yield {"text":t}

train_data = Dataset.from_generator(gen)
train_data = train_data.map(tokenize)


def gen2():
  for t in text_with_prompt_template[200:400]:
    yield {"text":t}

validation_data = Dataset.from_generator(gen2)
validation_data = validation_data.map(tokenize)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a very helpful assistant. Answer questions as best you can.<|eot_id|><|start_header_id|>user<|end_header_id|>

How can I evaluate the performance and quality of the generated text from Lamini models?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

There are several metrics that can be used to evaluate the performance and quality of generated text from Lamini models, including perplexity, BLEU score, and human evaluation. Perplexity measures how well the model predicts the next word in a sequence, while BLEU score measures the similarity between the generated text and a reference text. Human evaluation involves having human judges rate the quality of the generated text based on factors such as coherence, fluency, and relevance. It is recommended to use a combination of these metrics for a comprehensive evaluation of the model's performance.<|eot_id|><|end_of_text|>
<|begin_of_text|><|start_header_id|>system<|en

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a very helpful assistant. Answer questions as best you can.<|eot_id|><|start_header_id|>user<|end_header_id|>

Can Lamini generate human-readable explanations for the predictions made by a customized LLM?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Yes, Lamini can generate human-readable explanations for the predictions made by a customized LLM. Lamini provides a feature called "Explainability" which allows users to understand how the model arrived at a particular prediction. This feature generates explanations in natural language, making it easy for users to understand the reasoning behind the model's predictions.<|eot_id|><|end_of_text|>
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a very helpful assistant. Answer questions as best you can.<|eot_id|><|start_header_id|>user<|end_header_id|>

Is it possible to fine-tune Lamini on a specific dataset for dialogue generation?<|eot_id|>

In [ ]:
train_data

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})

In [ ]:
validation_data

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})

------------------------------
## Hyper-parameteres

Here we set all the hyper-parameters we need:
- specify the device to exploit the GPU (`cuda`)
- use the tokenizer for llama
- we set then the LoRA hyper-params:
  - the rank `R` of the two matrices A and B
  - the normalization factor `Alpha`
  - the `dropout rate`
  - the target modules, i.e. where to insert the LoRA modules: `q`, `k` and `v` for the attention and `o` for the final output layer.
- `number of epochs` to train the model
- `batch_size` is the global size of our batch of examples, but since we have a small GPU with only 15GB of memory, we need to scale this down, so we introduce 2 new concepts:
  - `micro_batch_size` is the real size of the batch we will use, which will be smaller (usually 2,4,8)
  - `gradient_accumulation_steps` how many steps (of batches) we want to accumulate the `loss` for before we update the parameters of the model. In this way we simulate a bigger `batch_size` by accumulating the `loss` for more than one iteration, and then we update the model.
- the `learning_rate`, which controls the intensity of the update
- the `warmup_ratio`, as we are using a scheduler for the learning rate, i.e. it will not be fixed during the whole training, but will vary based on the time.


We will be using a linear decay scheduler with warmup of `0.1`, as in the figure below. This means it will linearly grow for `10%` of the training, and the linearly decay for the rest.

<img src="https://i.stack.imgur.com/Ujhvo.png" width="450px">

In [ ]:

LORA_R = 8 #LORA MATRIX A and B RANK
LORA_ALPHA = 16 #SCALE FACTOR IN MULTIPLICATION
'''
Alpha is a scaling factor -- it changes how the adaptation layer's weights affect the base model's.
Higher alpha means the LoRA layers act more strongly on the base model.
A lot of the early LoRA examples (e.g. Alpaca-LoRA, etc) tend to use a low rank, like 8, 16, or 32, with roughly double that value for alpha
'''
LORA_DROPOUT= 0.05 #RANDOM WEIGHTS DROPOUT FOR AVOIDING OVERFITTING
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj", #MULTI-HEAD ATTENTION WEIGHTS + Output Layer
]

EPOCHS = 15
BATCH_SIZE = 32 #128 Is the prefered value (lower to fit in Colab)
MICRO_BATCH_SIZE = 8 #32 #4
GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE
LEARNING_RATE = 2e-4 #Standard learning rate
WARMUP_RATIO = 0.1

Now we need to load the model and its associated tokenizer. Choose here the number of bits for loading the models. Remember that lower bits mean lower precision, and thus a drop in performance.

In [ ]:
bits = "4" #@param ["4", "8", "full"]

In [ ]:
#-------------------
#   LOAD MODEL
#-------------------

from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
import torch
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained("1024m/Llama-3.2-3B-Instruct-Base", use_fast=False)

# base model here, choose between 4, 8 bits or full precision
if bits == "8":
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True
        # Removed the invalid 8-bit parameters!
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto"
    )
    # Prepare only if quantized
    model = prepare_model_for_kbit_training(model)

elif bits == "4":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto"
    )
    # Prepare only if quantized
    model = prepare_model_for_kbit_training(model)

else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map="auto"
    )

# ADD LORA MODULES
config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

# LLaMA doesn't have a default pad token, so we usually set it to the End-Of-Sequence (EOS) token
tokenizer.pad_token = tokenizer.eos_token

training_arguments = transformers.TrainingArguments(
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_ratio=WARMUP_RATIO,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001, # Weight decay to apply to all layers except bias/LayerNorm weights
    lr_scheduler_type = "linear",  # Linear decay of Learning Rate
    fp16=True,
    logging_strategy = "steps",
    logging_steps=1,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    output_dir=OUTPUT_DIR,
    save_total_limit=1,
    label_names=["labels"]
)

# Initialize the collator
# Swap out DataCollatorForLanguageModeling for Seq2Seq
data_collator = transformers.DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model, # Passing the model is optional but recommended for padding behavior
    padding=True,
    return_tensors="pt"
)

# istantiate a Trainer object using the hyper-params we defined earlier
trainer = transformers.Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=validation_data,
    args=training_arguments,
    data_collator=data_collator
)
model.config.use_cache = False

if torch.cuda.device_count() > 1:
    # keeps Trainer from trying its own DataParallelism when more than 1 gpu is available
    model.is_parallelizable = True
    model.model_parallel = True


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 4,587,520 || all params: 3,217,337,344 || trainable%: 0.1426


In [ ]:
#-------------------
#    TRAIN & SAVE
#-------------------
trainer.train()
model.save_pretrained(OUTPUT_DIR)

Epoch,Training Loss,Validation Loss
1,3.663152,3.151725
2,2.236090,2.233532
3,1.668920,1.699699
4,1.511209,1.508429
5,1.296796,1.421285
6,1.346666,1.381020
7,1.193203,1.357147
8,1.120781,1.340425
9,1.400157,1.328922
10,1.332541,1.319668


If you need to clear memory from the gpu, uncomment this code and run it.

**WARNING**: this will delete the `model` variable and delete anything was stored on the GPU!

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Reload model in FP16 and merge it with LoRA adapter weights - SAVE FULL MODEL
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "HCNLP-Llama-3.2-3B-Instruct-LLamini", offload_folder="offload/")
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained("1024m/Llama-3.2-3B-Instruct-Base", use_fast=False)

model.save_pretrained("HCNLP-Llama-3.2-3B-Instruct-LLamini_final")
tokenizer.save_pretrained("HCNLP-Llama-3.2-3B-Instruct-LLamini_final")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('HCNLP-Llama-3.2-3B-Instruct-LLamini_final/tokenizer_config.json',
 'HCNLP-Llama-3.2-3B-Instruct-LLamini_final/chat_template.jinja',
 'HCNLP-Llama-3.2-3B-Instruct-LLamini_final/tokenizer.json')

In [ ]:
#!pip install -U huggingface_hub

In [ ]:
# 1. Initialize the API
from huggingface_hub import HfApi
api = HfApi(token="YOUR_HF_WRITE_TOKEN")

# 3. Create the repository on Hugging Face (exist_ok=True means it won't crash if it's already there)
api.create_repo(repo_id="m-polignano/HCNLP-Llama-3.2-3B-Instruct-LLamini", exist_ok=True, repo_type="model")
api.upload_folder(repo_id="m-polignano/HCNLP-Llama-3.2-3B-Instruct-LLamini",folder_path="/content/HCNLP-Llama-3.2-3B-Instruct-LLamini_final",repo_type="model")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...i_final/model.safetensors:   0%|          | 31.4MB / 6.43GB            

  ...mini_final/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/m-polignano/HCNLP-Llama-3.2-3B-Instruct-LLamini/commit/682ada0eaf047a446c4dba7d839c01c866f0aa43', commit_message='Upload folder using huggingface_hub', commit_description='', oid='682ada0eaf047a446c4dba7d839c01c866f0aa43', pr_url=None, repo_url=RepoUrl('https://huggingface.co/m-polignano/HCNLP-Llama-3.2-3B-Instruct-LLamini', endpoint='https://huggingface.co', repo_type='model', repo_id='m-polignano/HCNLP-Llama-3.2-3B-Instruct-LLamini'), pr_revision=None, pr_num=None)

## Upload the model to Huggingface

If you want to share the model with the community (or just to share it between servers), you can push it to the HuggingFace hub by using:

```python
model.push_to_hub("my-awesome-model")
```

Or you can push it directly when saving, at the end of the training procedure:

```python
model.save_pretrained(OUTPUT_DIR, push_to_hub=True, repo_name="my-awesome-model")
```

Of course you first need to [create](https://huggingface.co/docs/hub/security-tokens) an API token from HuggingFace:
- go on your profile
- click on Edit Profile
- on the left you will find "Access Tokens"
- create a new token by pressing on "New Token"
- give it a name and make sure it's set on "write"

After this, just copy it and login using `huggingface-cli`




#Inference: generating answers

In this Notebook we will see how to use the model we trained earlier for generating answers on new data, maybe a test set. Here we will load it from Huggingface (but if you have saved it on your colab it's almost the same), we will se some examples and we will ask the model to answer.

In [ ]:
import torch
from transformers import pipeline

model_id = "1024m/Llama-3.2-3B-Instruct-Base"
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
messages = [
    {"role": "system", "content": "You are a very helpful assistant. Answer questions as best you can."},
    {"role": "user", "content": "What is the purpose of the `LLM` class in the Lamini Python package?"},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': "I'm not familiar with the specific Lamini Python package you're referring to. However, I can tell you about the LLaMA model, which is a popular large language model developed by Meta AI.\n\nLLaMA is not a Python class, but rather a pre-trained language model that can be used in various applications, including text generation, language translation, and more. It's a large transformer-based model that has been trained on a massive dataset of text.\n\nIf you're referring to a different package or implementation of LLaMA in Python, I'd be happy to try and help you further. Can you please provide more context or information about the Lamini Python package and its `LLM` class?"}


In [ ]:
import torch
from transformers import pipeline

model_id = "m-polignano/HCNLP-Llama-3.2-3B-Instruct-LLamini"
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
messages = [
    {"role": "system", "content": "You are a very helpful assistant. Answer questions as best you can."},
    {"role": "user", "content": "What is the purpose of the `LLM` class in the Lamini Python package?"},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': 'The `LLM` class in the Lamini Python package is used to interface with the Lamini LLM Engine. The LLM Engine is a powerful tool for training and fine-tuning language models, and the `LLM` class provides a simple and easy-to-use interface for working with the engine.'}


In [ ]:
import datasets
from datasets import load_dataset
#FORMAT DATA
instruction_dataset_df = load_dataset("kotzeje/lamini_docs.jsonl", split="train", streaming=True)

prompt_template_LLaMa = '''<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a very helpful assistant. Answer questions as best you can.<|eot_id|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
'''

text_with_prompt_template=[]
answers = []
questions = []
for row in instruction_dataset_df:
  question = row["question"]
  answer = row["answer"]
  questions.append(question)
  answers.append(answer)

  text_with_prompt_template.append(prompt_template_LLaMa.format(question=question))
print(text_with_prompt_template[0])
#No answers are included into the prompts



<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a very helpful assistant. Answer questions as best you can.<|eot_id|><|start_header_id|>user<|end_header_id|>

How can I evaluate the performance and quality of the generated text from Lamini models?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



In [ ]:
# inference
import numpy as np
from transformers import AutoTokenizer
genAnswers = []
z = 400
tokenizer = AutoTokenizer.from_pretrained("m-polignano/HCNLP-Llama-3.2-3B-Instruct-LLamini",fast=False)
with torch.no_grad():
  # LLaMA doesn't have a default pad token, so we usually set it to the End-Of-Sequence (EOS) token

  tokenizer.pad_token = tokenizer.eos_token

  for t in text_with_prompt_template[400:420]:
    tokenized_inputs = tokenizer(t, return_tensors="pt", padding=True, truncation=True).to(pipe.model.device)

    gen_outputs = pipe.model.generate(
      **tokenized_inputs,
      return_dict_in_generate=False,
      output_scores=False,
      max_new_tokens=128,
    )

    # decoding and printing
    for i in range(len(gen_outputs)):
      output = tokenizer.decode(gen_outputs[i], skip_special_tokens=True)
    print("Question:" + questions[z]+"\n")
    print("Real Answer:" + answers[z]+"\n")
    print("Predicted Answer:" + output.split("assistant\n")[-1])
    print("#########################################")
    genAnswers.append(output.split("assistant\n")[-1])
    z = z+1

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Question:Can Lamini have conversations with me, like a friend?

Real Answer:LLM Engine is a language model that can be used to generate responses to conversations. However, it is not designed to be a friend or a substitute for human interaction. Its purpose is to assist with specific tasks and provide helpful responses based on the input it receives.

Predicted Answer:No, Lamini is a language model that can only process text inputs and generate text outputs. It does not have the ability to engage in conversations or have personal interactions.
#########################################


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Question:Can you provide an example use case where Lamini outperforms ChatGPT in handling internal engineering documentation?

Real Answer:An example use case where Lamini outperforms ChatGPT is in handling internal engineering documentation. While ChatGPT with retrieval may lack sufficient context to synthesize information accurately, Lamini's LLM, trained on 100% of the Lamini documentation, possesses specific knowledge, can aggregate information from different sections of documentation, and steer the conversation back to relevant topics.

Predicted Answer:Lamini can be used to generate internal engineering documentation by providing a clear and concise description of the model's architecture and training process. This can help engineers understand the inner workings of the model and make informed decisions about its deployment and maintenance. In contrast, ChatGPT may struggle to provide detailed technical information about the model's architecture, as its primary focus is on genera

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Question:Does the documentation provide information on performance optimization or best practices for using the code?

Real Answer:Yes, the documentation has information on running a model using a batch interface as well as using a real-time interface. Besides that, the LLM Engine will optimize performance automatically.

Predicted Answer:No, the documentation does not provide information on performance optimization or best practices for using the code.
#########################################


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Question:Are there any mechanisms in Lamini to mitigate bias or fairness issues in LLM training?

Real Answer:Yes, Lamini provides mechanisms to mitigate bias and fairness issues in LLM training. One approach is to use techniques such as adversarial training or data augmentation to increase the diversity of the training data and reduce bias. Additionally, Lamini allows for fine-tuning of pre-trained models on specific domains or use cases, which can help to reduce bias and improve fairness. Finally, Lamini provides tools for analyzing and interpreting the behavior of LLMs, which can help to identify and address any bias or fairness issues that may arise during training.

Predicted Answer:Yes, Lamini provides mechanisms to mitigate bias and fairness issues in LLM training. These mechanisms include data preprocessing techniques, such as removing sensitive information and handling missing values, as well as algorithms that promote fairness and reduce bias, such as fairness-aware optimizat

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Question:Can you explain the process of adding data using the `add_data()` function? What formats are supported for training data?

Real Answer:The `add_data()` function in the `Program` class allows you to add training data to your program. It supports both singleton and list formats for the examples parameter. If the examples parameter is a list, related information can be grouped together. The function `value_to_dict()` is used to convert the examples to a dictionary format.

Predicted Answer:Adding data to a model using the `add_data()` function involves providing the training data in a specific format. The supported formats for training data include CSV and JSON. The data should be in a pandas DataFrame or a list of lists, where each inner list represents a row of data. The columns in the data should match the column names specified in the `add_data()` function. The data should also be properly formatted, with each row being a dictionary where the keys are the column names and the

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
scores = []
for i in range(len(genAnswers)):
  reference = [answers[i]]
  test = genAnswers[i]
  score = sentence_bleu(reference, test)
  scores.append(score)

print("Mean BLEU Score: "+str(sum(scores) / len(scores)))

Mean BLEU Score: 0.15953361320575232


In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"
!pip install rouge-score

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

scores = []
for i in range(len(genAnswers)):
  reference = answers[i]
  test = genAnswers[i]
  score = scorer.score(reference, test)['rougeL'][2] #f1measure
  scores.append(score)
mean = sum(scores) / len(scores)
print("Mean ROUGE-L Score: "+ str(mean))

Mean ROUGE-L Score: 0.0887621553965324


In [ ]:
import gc
del pipe
del tokenizer
gc.collect()
torch.cuda.empty_cache()

# Unsloth
## Installation

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the `Llama-3.2` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we convert it to HuggingFace's normal multiturn format `("role", "content")` instead of `("from", "value")`/ Llama-3 renders multi turn conversations like below:

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hey there! How are you?<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm great thanks!<|eot_id|>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3` and more.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

We now use `standardize_sharegpt` to convert ShareGPT style datasets into HuggingFace's generic format. This changes the dataset from looking like:
```
{"from": "system", "value": "You are an assistant"}
{"from": "human", "value": "What is 2+2?"}
{"from": "gpt", "value": "It's 4."}
```
to
```
{"role": "system", "content": "You are an assistant"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [ ]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

We look at how the conversations are structured for item 5:

In [ ]:
dataset[5]["conversations"]

[{'content': 'How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?',
  'role': 'user'},
 {'content': 'Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.',
  'role': 'assistant'}]

In [ ]:
print(dataset[5]["text"])

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAstronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.<|

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

Unsloth: Removed 693 out of 100000 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


We verify masking is actually done:

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAstronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight rel

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                  Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.<|eot_id|>'

We can see the System and Instruction prompts are successfully masked!

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
3.07 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 99,307 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.046300
2,0.950600
3,0.738300
4,1.019400
5,0.718700
6,1.045700
7,1.036300
8,0.835000
9,0.955800
10,1.141700


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

459.4731 seconds used for training.
7.66 minutes used for training.
Peak reserved memory = 4.342 GB.
Peak reserved memory for training = 1.272 GB.
Peak reserved memory % of max memory = 29.815 %.
Peak reserved memory for training % of max memory = 8.734 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContinue the fibonacci sequence: 1, 1, 2, 3, 5, 8,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe Fibonacci sequence is a series of numbers in which each number is the sum of the two preceding numbers. The sequence is: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144,']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

1, 13, 21, 34, 55, 89.<|eot_id|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

('llama_lora/tokenizer_config.json',
 'llama_lora/special_tokens_map.json',
 'llama_lora/chat_template.jinja',
 'llama_lora/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
The Eiffel Tower, located in Paris, the capital of France, is a towering iron lattice structure that stands 324 meters (1,063 feet) tall. It is an iconic symbol of the city and a major tourist attraction. The Eiffel Tower was built for the 1889 World's Fair, held in Paris, and its construction required the development of advanced engineering techniques to support its height. It was intended to be a temporary structure, but it has become a permanent part of the Parisian skyline and a cultural landmark around the world.<|eot_id|>


You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("llama_lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>

# Unsloth GRPO

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

Goal: To convert `Llama-3.2-1B-Instruct` into a reasoning model via GRPO by using OpenR1's Math dataset.

We first pre fine-tune the model to make GRPO skip trying to match formatting - this speeds GRPO up.

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 02-20 17:55:37 [__init__.py:244] Automatically detected platform cuda.
ERROR 02-20 17:55:41 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-20 17:55:56 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
INFO 02-20 17:55:56 [vllm_utils.py:752] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM l

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 02-20 17:56:39 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 02-20 17:56:39 [config.py:1472] Using max model len 1024
WARNING 02-20 17:56:39 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 02-20 17:56:45 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 02-20 17:56:45 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='unsloth/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='unsloth/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto'

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth 2026.2.1 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [ ]:
messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 512, use_cache = True, streamer = TextStreamer(tokenizer))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 20 Feb 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve x^5 + 3x^4 - 10 = 3.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

To solve the equation x^5 + 3x^4 - 10 = 3, we need to isolate the variable x.

First, let's add 10 to both sides of the equation:

x^5 + 3x^4 = 3 + 10
x^5 + 3x^4 = 13

Next, let's subtract 13 from both sides of the equation:

x^5 + 3x^4 - 13 = 0

Now, we have a polynomial equation of degree 5. Unfortunately, this equation does not have a simple closed-form solution. However, we can try to find the roots using numerical methods or graphing.

Using numerical methods, we can find that one of the roots of the equation is approximately x = 1.324.

To find the other roots, we can use numerical methods such as the Newton-Raphson method or graphing. However, these methods are not provided here as they are complex and require a significant 

Let's call the model as is:

In [ ]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 512,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'?\nA) 10\nB) 11\nC) 16\nD) 15.61\nThe best answer is D.'

### GRPO chat template
Since we're using a base model, we should set a chat template. You can make your own chat template as well!
1. DeepSeek uses `<think>` and `</think>`, but this is **not** necessary - you can customize it however you like!
2. A `system_prompt` is recommended to at least guide the model's responses.

In [ ]:
reasoning_start = "<start_working_out>" # Acts as <think>
reasoning_end   = "<end_working_out>"   # Acts as </think>
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>'

We create a simple chat template below. Notice `add_generation_prompt` includes prepending `<start_working_out>` to guide the model to start its reasoning process.

In [ ]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with our specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

Let's see how our chat template behaves on an example:

In [ ]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|eot_id|>What is 1+1?<start_working_out>I think it's 2.<end_working_out><SOLUTION>2</SOLUTION><|eot_id|>What is 2+2?<start_working_out>"

### Pre fine-tuning for formatting
We now use a subset of NVIDIA's [Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning) which was filtered to only include high quality DeepSeek R1 traces.

We'll only filter ~59 or so examples to first "prime" / pre fine-tune the model to understand our custom GRPO formatting.

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# Select only numbers
dataset = dataset.iloc[np.where(is_number)[0]]

dataset

,expected_answer,problem,generated_solution
0,14,Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$...,"<think>\nOkay, let's see. I need to solve the ..."
6,-2,Find the value of the parameter $a$ for which ...,"<think>\nOkay, so I need to find the value of ..."
9,18,What is the sum of all real numbers $x$ for wh...,"<think>\nOkay, so I need to solve the equation..."
13,2,Evaluate the sum \(\sum_{n=1}^\infty \frac{\ph...,"<think>\nOkay, so I need to evaluate the infin..."
17,30,What is the largest positive integer that divi...,"<think>\nAlright, so I need to find the larges..."
...,...,...,...
19243,244,"Let \( p \), \( q \), and \( r \) be the disti...","<think>\nOkay, so I need to find the value of ..."
19245,1,A bug is on the $0$ of a number line. At any p...,"<think>\nOkay, so I have this problem where a ..."
19247,4,A bus left point X for point Y. Two hours late...,"<think>\nOkay, let's tackle this problem step ..."
19248,18,Each interior angle of a regular n-gon measure...,"<think>\nOkay, let's see. I need to find the n..."


We have to format the dataset to follow our GRPO style formatting:

In [ ]:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)

Check to see if it worked:

In [ ]:
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|eot_id|>Given $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<start_working_out>Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 165

Let's truncate the pre fine-tuning dataset to `max_seq_length/2` since we don't want too long reasoning traces.

Note this might take 2 minutes!

In [ ]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

We then tokenize the messages and convert it to a Hugging Face compatible dataset format:

In [ ]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

Dataset({
    features: ['expected_answer', 'problem', 'generated_solution', 'Messages', 'N', 'text', '__index_level_0__'],
    num_rows: 7507
})

In [ ]:
dataset["text"][0]

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|eot_id|>Given $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<start_working_out>Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 165

Let's now pre fine-tune the model so it follows our custom GRPO formatting!

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 10,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 100,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/7507 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,507 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,1.255300
10,1.128700
15,1.100700
20,0.963200
25,0.936400
30,0.921000
35,0.934500
40,0.910000
45,0.962100
50,0.852600


TrainOutput(global_step=100, training_loss=0.9305826616287232, metrics={'train_runtime': 187.2159, 'train_samples_per_second': 2.137, 'train_steps_per_second': 0.534, 'total_flos': 2446363755528192.0, 'train_loss': 0.9305826616287232, 'epoch': 0.05327650506126798})

Let's check if the model has learnt to follow the custom format:

In [ ]:
import gc
for _ in range(5):
    torch.cuda.empty_cache()
    gc.collect()

Yes it did follow the formatting! Great! Let's remove some items before the GRPO step

In [ ]:
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()

0

### Data Prep
<a name="Data"></a>

We're using Hugging Face's [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed). You can also utilize OpenAI's famous [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

README.md: 0.00B [00:00, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14116 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 14116
})

Let's look at the first row:

In [ ]:
dataset[0]["prompt"]

'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.'

In [ ]:
dataset[0]["solution"]

'34'

In GSM8K, we notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

In [ ]:
def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text
extract_hash_answer(dataset[0]["solution"])

'34'

Let's map the dataset! and see the first row:

In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

{'prompt': [{'content': 'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>',
   'role': 'system'},
  {'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.',
   'role': 'user'}],
 'solution': '34',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $

We create a regex format to match the reasoning sections and answers:

In [ ]:
import re

# Add optional EOS token matching
solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"

match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
match_format

re.compile(r'<end_working_out>.*?<SOLUTION>(.+?)</SOLUTION>[\s]{0,}(?:<\|eot_id\|>)?[\s]{0,}$',
re.MULTILINE|re.DOTALL|re.UNICODE)

We verify it works:

In [ ]:
match_format.findall(
    "Let me think!<end_working_out>"\
    f"<SOLUTION>\n2\n</SOLUTION>",
)

['\n2\n']

In [ ]:
match_format.findall(
    "<start_working_out>Let me think!<end_working_out>"\
    f"<SOLUTION>  2  </SOLUTION>\n\n",
)

['  2  ']

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [ ]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!

        # No need to reward <start_working_out> since we always prepend it!
        # score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores

Finally, we want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # Penalize wrong answers
            except:
                score -= 4.5 # Penalize
        scores.append(score)
    return scores

Also sometimes it might not be 1 number as the answer, but like a sentence for example "The solution is $20" -> we extract 20.

We also remove possible commas for example as in 123,456

In [ ]:
match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

['0.34']
['123,456']
['-0.234']
['17']


We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores

Get the top 90% prompt length so we don't accidentally truncate them!

Ie we'll remove the top 10% long prompts.

In [ ]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|eot_id|>In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outside triangle $ABC$ such that $\angle BAD = \angle DAC$ and $\angle BDC = 90^\circ$. Suppose that $AD = 1$ and that $\frac{BD}{CD} = \frac{3}{2}$. If $AB + AC$ can be expressed in the form $\frac{a\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.<start_working_out>


Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

Max Length =  198


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,715 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)


Unsloth: Will smartly offload gradients to save VRAM!
********************Question:
A conical glass is in the form of a right circular cone. The slant height is $21$ and the radius of the top rim of the glass is $14$. An ant at the midpoint of a slant line on the outside wall of the glass sees a honey drop diametrically opposite to it on the inside wall of the glass. If $d$ is the shortest distance it should crawl to reach the honey drop, what is the integer part of $d$? 
Answer:
18 
Response:
Okay, let's see. So the problem involves a conical glass, and there's an ant sitting at the midpoint of a slant line on the outside wall of the glass. It needs to find the shortest distance, d, it should crawl to reach a honey drop diametrically opposite to it on the inside wall. The question is asking for the integer part of d. Right. So, let me imagine how to approach this problem step by step.

First, let me recall some geometry concepts that might help here. Since the glass is a right circula

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / check_answer / mean,rewards / check_answer / std,rewards / check_numbers / mean,rewards / check_numbers / std
1,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.280324,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
2,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.303143,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
3,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.259834,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
4,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.346491,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
5,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.286126,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
6,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.293295,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
7,0.000200,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.234861,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
8,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.272787,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
9,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.281671,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
10,0.000300,-7.500000,0.000000,825.000000,825.000000,825.000000,1.000000,0.000000,0.000000,0.000000,0.286182,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000


********************Question:
Find the number of prime numbers $p$ between $100$ and $200$ for which the congruence equation $x^{11} + y^{16} \equiv 2013 \pmod{p}$ has a solution in integers $x$ and $y$. 
Answer:
21 
Response:
Okay, so I need to find how many prime numbers p between 100 and 200 (not just over 100, but strictly between 100 and 200) have solutions to the congruence equation x^{11} + y^{16} ≡ 2013 modulo p. Hmm, that's a bit complicated. Let's break it down step by step.

First, the problem states that we need to find the number of prime numbers p between 100 and 200 for which this congruence has a solution in integers x and y. So, I need to check all primes between 100 and 200, compute the polynomial equation x^{11} + y^{16} ≡ 2013 modulo p for each prime p, and then see how many times the solution wraps around (i.e., the generated integers x and y give the same value modulo p when x ≡ y mod p) for any two different x and y where the original values x and y satisfy the c

TrainOutput(global_step=100, training_loss=0.006751779992046067, metrics={'train_runtime': 1696.9428, 'train_samples_per_second': 0.236, 'train_steps_per_second': 0.059, 'total_flos': 0.0, 'train_loss': 0.006751779992046067})

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.3,
    top_k = 50,
    max_tokens = 1024,
    repetition_penalty=10
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

' \nsqrt(100) =10\nThe square root function takes a number as input and returns its value when raised to half. For example, if you have two numbers that are equal (i.e., they both end with zero), their squares will be identical.\nSo,\n√25=5 because it\'s five squared which equals twenty-five.\n\nIn general:\n- √16=(4)\nSince we know what an integer’s natural exponent means in terms for our current problem: If x^y=x^(1/y)=x^{log(x)/ log(y)}, then taking logarithms can help us find answers like this one easily!\nHere let me show how I would do something similar using Python code:\n\n```python\n\nimport math\n\n\n# Define variables before use:\n\n\ndef calculate_sqrt(n):\n    # Calculate Square Root Using Math Formula or Built-in Function\n\n\n\nresult=sqrt(math.sqrt(m))\nreturn result\n\n\n\n\nprint(calculate.Sqrt("102"))\n\n\n\n\n\n```\n\nWhen run on "103", output should return `11`. This shows my method works!\n\nI hope your explanation helped clarify things! Let feedback about any que

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test:

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.3,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Okay, let's try to figure out the square root of 101. Hmm, I need to find the number that, when multiplied by itself, gives 101. That's a pretty straightforward problem. Let me think step by step.\n\nFirst, I remember that the square root of a number is a value that, when multiplied by itself, gives the original number. For example, the square root of 16 is 4 because 4 multiplied by 4 is 16. Similarly, the square root of 25 is 5 because 5 multiplied by 5 is 25. So, the square root of a number is the number itself if the number is a perfect square. But in this case, 101 is not a perfect square. So, I need to find the square root of 101. Let me try to factor 101 into two numbers that when multiplied together give 101. Maybe I can use some prime numbers or factorization techniques.\n\nWait, let me recall that 101 is a prime number. That means it's only divisible by 1 and itself. So, if I can express 101 as a product of two numbers, one of which is 1 and the other is 101, then that would 

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

#Evaluate on MMLU Benchmark using LM-EVAL. Around 20H of computation

In [ ]:
!pip install git+https://github.com/EleutherAI/lm-evaluation-harness.git@big-refactor

In [ ]:
!lm_eval --model hf-auto --model_args pretrained=unsloth/Llama-3.2-3B-Instruct,dtype="float16",load_in_4bit=True --tasks mmlu --device cuda:0